## Basic RAG
End to End Agentic RAG system

In [1]:
from dotenv import load_dotenv

from langchain_groq import ChatGroq

load_dotenv()

True

In [13]:
llm = ChatGroq(
	model="openai/gpt-oss-120b",
	temperature=0.0
)

In [14]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/mnt/Scratch/codebase/rag-projects/optirag/data/Stock_Market_Performance_2024.pdf")

documents = loader.load()

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
	chunk_size=500,
	chunk_overlap=150
)

chunks = splitter.split_documents(documents)

In [16]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 662.09it/s]


In [17]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
	documents=chunks,
	embedding=embedding_model,
	persist_directory="./chroma_db"
)

In [18]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [19]:
from langchain_core.tools import tool

@tool
def search_document_tool(query: str) -> str:
	"""Search the document and returns the output as per the query input"""
	
	docs = retriever.invoke(query)

	if not docs:
		return "No relevant information found."

	return "\n\n".join(doc.page_content for doc in docs)

In [20]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Stock Market Annual Report Analysis Agent
assistant = create_agent(
    model=llm,
    tools=[search_document_tool],
    system_prompt="""
    You are a Stock Market and Financial Statement Analysis Expert.

    Analyze the company's annual report to evaluate its financial
    health, business performance, growth potential, and investment risks.

    Search relevant sections, including:
    - Income Statement, Balance Sheet, and Cash Flow Statement
    - Management Discussion & Analysis (MD&A)
    - Revenue, Profit, EPS, and other financial metrics
    - Debt, liquidity, and cash flow position
    - Business segments and geographical performance
    - Risk Factors and Corporate Governance
    - Auditor's Report and Notes to Accounts
    - Management's future outlook and growth strategy
    - Dividend policy and capital allocation

    Use only information available in the annual report.
    Support every conclusion with evidence and cite page numbers or
    sections whenever available.
    Clearly distinguish reported facts from analytical interpretations.
    Do not invent financial figures or make unsupported stock predictions.
    If information is unavailable, explicitly state that.
    """,
)

# Stock Market Annual Report Analysis Task
task = """
Analyze the company's annual report and prepare a stock market
investment research report answering the following questions:

1. Financial Performance
   - How did revenue, net profit, operating profit, and EPS change?
   - What are the key profitability and growth trends?

2. Financial Health
   - What is the company's debt position and debt-to-equity ratio?
   - How strong are its operating cash flow and liquidity?
   - Is the company generating sustainable cash flows?

3. Business Performance
   - What are the major business highlights and segment-wise results?
   - What are the company's competitive advantages and growth drivers?
   - What challenges affected its performance?

4. Management and Strategy
   - What are management's future growth plans and outlook?
   - What are the planned capital expenditures and expansion initiatives?
   - How credible is management's outlook based on reported performance?

5. Risks
   - What are the major business, financial, regulatory, and operational risks?
   - Are there any auditor qualifications, governance concerns, or contingent liabilities?

6. Shareholder Returns
   - What is the dividend policy and dividend per share?
   - Has the company issued shares, repurchased shares, or diluted equity?
   - What does the report disclose about shareholder value creation?

7. Investment Analysis
   - What are the company's key strengths and weaknesses?
   - What positive and negative factors could influence its future performance?
   - What financial indicators should investors monitor in future reports?

Expected Output:
Prepare a structured investment research report with headings, tables
where useful, important financial metrics, year-over-year comparisons,
key business developments, growth opportunities, major risks, and
evidence with page numbers or report sections.

Use only the annual report. Do not invent missing figures, market prices,
valuation multiples, or future returns. Clearly state when information
cannot be determined from the document.
"""

# Execute the agent
response = assistant.invoke({
    "messages": [HumanMessage(content=task)]
})

result = response["messages"][-1].content

print("\n" + "=" * 80)
print("STOCK MARKET ANNUAL REPORT ANALYSIS")
print("=" * 80)
print(result)


STOCK MARKET ANNUAL REPORT ANALYSIS
**Important Note – Insufficient Source Material**

The analysis you have requested requires detailed quantitative and qualitative information that is normally found in a company’s annual report (financial statements, MD&A, risk factors, governance disclosures, etc.).  

I have attempted to locate the relevant sections of an annual report using the available search tool, but the queries returned only generic, unrelated text fragments (e.g., references to “Meta” or “Alphabet”) and no structured data that can be tied to a specific company’s filing. Consequently, I cannot extract:

* Revenue, net profit, operating profit, EPS, or any year‑over‑year changes  
* Balance‑sheet items such as total debt, debt‑to‑equity, cash and cash equivalents, or liquidity ratios  
* Segment‑wise performance, management’s outlook, capital‑expenditure plans, risk‑factor disclosures, dividend policy, share‑issuance or repurchase activity, auditor’s report, or any other requ